# 04 · SAM2 미세조정 — LoRA 로 우리 모델 만들기

> **공부 기록 노트북 4번 — 이 프로젝트의 주력 모델.** 03 에서 SAM2 가
> *프롬프트를 주면* 잘 한다는 걸 봤습니다. 이번엔 SAM2 를 우리 데이터에
> 맞게 *살짝 학습*시켜서, **프롬프트 없이 6-class 를 직접** 내놓게 만듭니다.

## 03 복습 — zero-shot 의 두 한계

1. **프롬프트가 있어야** 동작 (매번 클릭 불가 → 자동화 안 됨)
2. **해부 지식이 없음** (우리 6 클래스를 모름)

## 이 노트북에서 할 일

1. **LoRA** 가 무엇이고 왜 쓰는지 (거대 모델을 *작게* 손보는 법)
2. SAM2 mask decoder 를 **프롬프트 없는 6-class 출력기**로 바꾸는 아이디어
3. `SAM2LoRASegmenter` 를 만들고 **전체의 몇 %만 학습**하는지 확인
4. 프롬프트 없는 forward 로 `(B, 6, H, W)` 가 나오는 것 확인
5. 미니 학습으로 loss 가 줄어드는 것 확인
6. 진짜 학습·비교는 어디서 하는지

⚠️ **GPU 런타임 필요** (SAM2 학습은 무겁습니다 — T4 기준).

## 1. zero-shot 의 한계를 넘는 법 — LoRA

해결책은 **우리 데이터로 살짝 학습(fine-tune)** 하는 것. 그런데 SAM2 는
~8000 만 파라미터의 거대 모델이라, 통째로 학습시키면:
- GPU 메모리 폭발
- 데이터가 적으면 과적합(overfitting)
- SAM2 가 원래 가진 *일반 지식*을 망가뜨릴 수 있음 (catastrophic forgetting)

### LoRA (Low-Rank Adaptation) — 비유

> 비싼 **기성복**(SAM2)이 거의 잘 맞습니다. 통째로 새로 짓지(full fine-tune)
> 말고, **소매·기장만 살짝 수선**하면 돼요. 원단(원본 가중치)은 그대로 두고
> *작은 수선 조각*만 새로 답니다.

LoRA 는 거대한 가중치 행렬 옆에 **작은 행렬 2 개**를 덧붙이고, *그 작은 것만*
학습합니다. 원본은 freeze (얼림).

```
   원본 가중치 W  (고정 · 거대)
        +
      A x B       (학습 · 아주 작음)   <- 이것만 업데이트
        =
   적응된 가중치
```

→ 학습 파라미터가 *수십 분의 1*. 메모리 적고, 과적합 덜하고, 원본 지식 보존.

> 키워드: fine-tuning, LoRA, low-rank adaptation, PEFT, overfitting

## 2. 프롬프트 없이 6-class 를 직접 — mask decoder 재활용

SAM2 의 mask decoder 는 원래 "프롬프트에 따라 마스크 1 개"를 내도록 설계됐지만,
*여러 마스크를 동시에* 내는 기능(multimask)이 있습니다. 우리는 이걸
**클래스 수(6)만큼** 내도록 바꿔서, 프롬프트 없이 **6 채널 = 6 클래스** 를
한 번에 출력하게 만듭니다.

- 인코더: **LoRA 만** 학습 (대부분 freeze)
- mask decoder: 6-class 용으로 **통째 학습** (작아서 부담 적음)

이게 `src/models/sam2_lora.py` 의 `SAM2LoRASegmenter` 입니다. 직접 만져 봅시다.

> 키워드: SAM2 mask decoder, multimask output, dense prediction

## 0. 환경 준비

이전 노트북들과 동일.

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 → Runtime > Change runtime type > GPU 로 바꾸세요")

### (선택) Google Drive 연결 — 다운로드·체크포인트 보존

Colab은 런타임이 초기화되면 `/content`가 비워져 데이터(3GB+)와 학습 체크포인트가
사라집니다. 아래 셀은 Drive를 마운트해 `data/`·`outputs/`·모델 캐시를 Drive에
저장하므로, **다음에 다시 와도 받아둔 데이터와 학습 결과를 그대로** 씁니다.

**기본값은 `False`(Drive 미사용)** — 전부 `/content` 에서 돌아갑니다. Drive 에 보존하고 싶을 때만 `USE_DRIVE = True` 로 바꾸세요(인증 창이 뜹니다).

In [ ]:
USE_DRIVE = False  # Drive 에 데이터·체크포인트를 보존하려면 True (Drive 없으면 그대로 두세요)
if USE_DRIVE:
    from src.utils.colab import mount_drive
    mount_drive()

## 3. 모델 만들기 — LoRA 의 효율 확인

`SAM2LoRASegmenter` 를 만들고, **전체 파라미터 중 몇 %만 학습**하는지 봅니다.
이 비율이 LoRA 의 핵심 — *작은 일부*만 학습합니다.

(SAM2 체크포인트 ~330MB 다운로드. 03 에서 받았으면 캐시에서 바로 옵니다.)

In [ ]:
import torch
from src.models.sam2_lora import SAM2LoRASegmenter

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SAM2LoRASegmenter(num_classes=6).to(device)   # pretrained=True (기본)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 파라미터   : {total / 1e6:6.1f} M")
print(f"학습할 파라미터 : {trainable / 1e6:6.1f} M   ({100 * trainable / total:.1f}%)")
print(f"-> 약 {100 * (1 - trainable / total):.0f}% 는 freeze (SAM2 의 일반 지식 보존)")

## 4. 프롬프트 없는 forward

03 에서는 SAM2 에 점을 찍어줘야 했죠. 이제는 *이미지만* 넣으면 바로
`(배치, 6, 높이, 너비)` 로짓이 나옵니다 — 클릭 불필요.

(아직 학습 전이라 mask decoder 가 무작위 → 결과는 엉망입니다. 하지만 *형태*가
핵심: 6 채널이 프롬프트 없이 나온다.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES
from src.data.transforms import build_eval_transforms

IMG = 512   # 모델이 내부에서 1024 로 키워 SAM2 에 넣습니다
val_ds = CholecSeg8kDataset(split="val", image_size=IMG,
                            transform=build_eval_transforms(IMG))
fixed = val_ds[0]
fixed_image = fixed["image"].unsqueeze(0).to(device)   # (1, 3, 512, 512)
fixed_mask = fixed["mask"]

model.eval()
with torch.no_grad():
    out = model(fixed_image)
print("입력:", tuple(fixed_image.shape), "→ 출력:", tuple(out.shape))
print("프롬프트(클릭/박스) 없이 바로 6-class 로짓이 나온다!")
pred_before = out.argmax(dim=1)[0].cpu()

## 5. 미니 학습 — encoder/decoder 학습률 분리

02 의 학습 루프와 똑같습니다 (forward → loss → backward → step). 한 가지 추가:
LoRA 어댑터(인코더)와 mask decoder 는 **다른 학습률**을 씁니다 — `param_groups`
가 둘을 나눠줍니다 (인코더는 살짝, 디코더는 더 크게).

⚠️ T4 에서 SAM2 학습은 **무겁습니다** (한 스텝 수 초). 여기선 24 스텝만 —
*메커니즘 확인용*입니다. (메모리 부족(OOM)이 나면 런타임 재시작 후 이 셀의
`STEPS` 를 줄이세요.)

In [ ]:
from torch.utils.data import DataLoader, Subset

from src.data.transforms import build_train_transforms
from src.losses.dice import DiceLoss
from src.losses.focal import FocalLoss

demo = Subset(
    CholecSeg8kDataset(split="train", image_size=IMG,
                       transform=build_train_transforms(IMG)),
    range(24),
)
loader = DataLoader(demo, batch_size=1, shuffle=True, num_workers=2)

dice, focal = DiceLoss(), FocalLoss(gamma=2.0)
optimizer = torch.optim.AdamW(
    model.param_groups(lr_encoder=1e-4, lr_decoder=1e-3))

STEPS = 24
model.train()
for step, batch in enumerate(loader):
    if step >= STEPS:
        break
    images = batch["image"].to(device)
    masks = batch["mask"].to(device)

    logits = model(images)                              # 1. forward
    loss = focal(logits, masks) + dice(logits, masks)   # 2. loss
    optimizer.zero_grad()
    loss.backward()                                     # 3. backward
    optimizer.step()                                    # 4. step
    if step % 4 == 0:
        print(f"step {step:2d}  loss {loss.item():.4f}")
print("미니 학습 완료")

## 6. 학습 전/후

24 스텝은 SAM2 에겐 *턱없이 부족*합니다 (진짜 학습은 수천 스텝). 큰 변화를
기대하진 마세요 — **loss 가 줄었고, 출력이 6-class 형태로 나온다**는 게 핵심.

In [ ]:
model.eval()
with torch.no_grad():
    pred_after = model(fixed_image).argmax(dim=1)[0].cpu()

img = fixed_image[0].cpu().permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(img);                                       ax[0].set_title("input")
ax[1].imshow(fixed_mask, vmin=0, vmax=5, cmap="tab10");  ax[1].set_title("ground truth")
ax[2].imshow(pred_before, vmin=0, vmax=5, cmap="tab10"); ax[2].set_title("before training")
ax[3].imshow(pred_after, vmin=0, vmax=5, cmap="tab10");  ax[3].set_title("after 24 steps")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 7. 진짜 학습과 비교는 어디서?

이 노트북은 *개념과 메커니즘*을 보는 곳입니다. 실제로 쓸 만한 모델을 얻으려면:

- **전체 학습**: `!python -m src.train.train_segmentation`
  (또는 `run_pipeline.ipynb`) — 수천 스텝, 체크포인트 저장, 중단 시 재개.
- **엄밀한 비교**: `!python -m src.eval.benchmark_runner` — U-Net vs SAM2+LoRA
  를 테스트 세트에서 mIoU · Cystic Duct Dice + 95% 부트스트랩 신뢰구간으로
  비교. 그 표가 README 의 결과표를 채웁니다.

즉 *공부는 이 노트북*, *결과는 run_pipeline + benchmark* — 역할이 나뉩니다.

## 마무리 — 이 노트북 정리

### 배운 것
- 거대 모델을 통째로 학습하면 메모리·과적합·지식손상 문제 → **LoRA** 로
  *작은 어댑터만* 학습하고 원본은 freeze. (기성복 수선 비유)
- 실제로 학습 파라미터가 전체의 극히 일부임을 param 카운트로 확인.
- SAM2 mask decoder 의 multimask 를 6-class 로 재활용 → **프롬프트 없이**
  `(B, 6, H, W)` 를 직접 출력.
- encoder(LoRA) 와 decoder 는 학습률을 분리 (`param_groups`).

### 아직 이해가 덜 된 것 / 더 파볼 것
- LoRA 의 rank(작은 행렬 크기)는 어떻게 정하나? →
  `configs/model/sam2_lora.yaml` 의 `lora.rank`, `src/models/sam2_lora.py`.
- mask decoder 를 통째 학습하는 게 맞나, 여기도 LoRA 가 나을까?
- 제대로 학습하면 U-Net 베이스라인을 이길까? → 전체 학습 후 benchmark 로 확인.

### 다음 노트북
**`05_cvs_classifier.ipynb`** — 분할은 끝. 이제 *두 번째 작업*: 분할 마스크
(6 채널) + 원본 프레임(3 채널) = **9 채널**을 ViT 분류기에 넣어 **CVS 3 기준**
을 판정하고 0~3 점을 낸다. 새 데이터셋 **Endoscapes2023** 도 등장한다.